In [ ]:
!pip install langchain langchain-groq langchain-community dotenv langchain-text-splitters langchain-huggingface sentence_transformers pypdf bs4 unstructured unstructured[pdf]

In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
import os
groq_key = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_community.document_loaders import TextLoader

document = TextLoader("./data/GTB_gold_Nov23.txt", encoding="utf-8").load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunks_spliter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = chunks_spliter.split_documents(document)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})

In [ ]:
from langchain_community.vectorstores import InMemoryVectorStore

vectorestore = InMemoryVectorStore.from_documents(
    documents=document,
    embedding=embeddings_model,
)

In [ ]:
retriever = vectorestore.as_retriever(serch_kwargs={"k": 2})

In [ ]:
similar_chunks = retriever.invoke(query)

similar_chunks

In [ ]:
similar_texts = [chunk.page_content for chunk in similar_chunks]

similar_texts

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Responda usando exclusivamente os conteudo fornecido. \n\nContexto:\n{context}"),
    ("human", "{query}")
])

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    groq_api_key=groq_key
)



In [ ]:
chain = prompt | model | StrOutputParser()

results = retriever.invoke(query)

context = "\n\n".join(result.page_content for result in results)

chain.invoke({"query": query, "context": context})

In [ ]:
from langchain_core.globals import set_debug

set_debug(True)

chain.invoke({"query": query, "context": context})

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PyPDFLoader("./data/GTB_gold_Nov23.pdf").load()

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://forbes.com.br/forbes-tech/2025/08/o-que-explica-o-fracasso-do-chatgpt-5-e-como-a-openai-vai-reagir/"

WebBaseLoader(web_path=url).load()